In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

# Fix random seed so results are reproducible every run
np.random.seed(42)

print("✅ Libraries imported")

✅ Libraries imported


In [2]:
DB_USERNAME = "root"
DB_PASSWORD = "password"
DB_HOST     = "127.0.0.1"
DB_PORT     = "3306"
DB_DATABASE = "bloodbridge_db"

engine = create_engine(
    f"mysql+pymysql://{DB_USERNAME}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_DATABASE}"
)

with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) as count FROM donors"))
    row    = result.fetchone()
    print(f"✅ Connected — Donors in DB: {row.count}")

✅ Connected — Donors in DB: 25


In [4]:
with engine.connect() as conn:
    real_df = pd.read_sql(text("""
        SELECT
            d.id                                                         AS donor_id,
            COUNT(rr.id)                                                 AS total_responses,
            COUNT(CASE WHEN rr.status IN (1,3) THEN 1 END)              AS accepted_count,
            COUNT(CASE WHEN rr.status = 5 THEN 1 END)                   AS no_show_count,
            COUNT(CASE WHEN rr.status = 2 THEN 1 END)                   AS declined_count,
            COUNT(CASE WHEN rr.status = 4 THEN 1 END)                   AS ignored_count,
            COALESCE(DATEDIFF(NOW(), MAX(rr.responded_at)), 999)         AS days_since_last,
            COALESCE(dhp.total_donations, 0)                             AS total_donations,
            COALESCE(dhp.blood_type, 0)                                  AS blood_type,
            TIMESTAMPDIFF(YEAR, d.birth_date, NOW())                     AS age,
            CASE d.gender WHEN 'male' THEN 0 ELSE 1 END                 AS gender
        FROM donors d
        LEFT JOIN request_responses rr
               ON d.id = rr.donor_id
              AND rr.status IN (1,2,3,4,5,6,7)
              AND rr.responded_at IS NOT NULL
        LEFT JOIN donor_health_profiles dhp ON d.id = dhp.donor_id
        WHERE d.deleted_at IS NULL
        GROUP BY d.id, dhp.total_donations, dhp.blood_type, d.birth_date, d.gender
    """), conn)

print(f"✅ Real data fetched: {len(real_df)} donors")
print(real_df.head())

✅ Real data fetched: 25 donors
   donor_id  total_responses  accepted_count  no_show_count  declined_count  \
0         1                0               0              0               0   
1         2                0               0              0               0   
2         3                0               0              0               0   
3         4                1               1              0               0   
4         5                1               1              0               0   

   ignored_count  days_since_last  total_donations  blood_type  age  gender  
0              0              999              5.0         1.0   35       0  
1              0              999              2.0         3.0   37       0  
2              0              999              3.0         5.0   31       0  
3              0                1              4.0         7.0   32       0  
4              0                1              7.0         2.0   34       0  


In [5]:
print("=== Real Data Statistics ===")
print(f"Total donors:          {len(real_df)}")
print(f"With responses:        {(real_df['total_responses'] > 0).sum()}")
print(f"Without responses:     {(real_df['total_responses'] == 0).sum()}")
print(f"Average acceptance:    {real_df['accepted_count'].sum() / max(real_df['total_responses'].sum(), 1):.2%}")
print(f"Average age:           {real_df['age'].mean():.1f} years")
print()
print(real_df.describe())

=== Real Data Statistics ===
Total donors:          25
With responses:        16
Without responses:     9
Average acceptance:    47.06%
Average age:           32.9 years

        donor_id  total_responses  accepted_count  no_show_count  \
count  25.000000        25.000000       25.000000      25.000000   
mean   13.000000         0.680000        0.320000       0.080000   
std     7.359801         0.556776        0.476095       0.276887   
min     1.000000         0.000000        0.000000       0.000000   
25%     7.000000         0.000000        0.000000       0.000000   
50%    13.000000         1.000000        0.000000       0.000000   
75%    19.000000         1.000000        1.000000       0.000000   
max    25.000000         2.000000        1.000000       1.000000   

       declined_count  ignored_count  days_since_last  total_donations  \
count       25.000000      25.000000        25.000000        25.000000   
mean         0.080000       0.120000       361.880000         3.0800

In [6]:
n = 500  # عدد السجلات

# ============================================================
# Step 1: Generate donor features
# Based on real data statistics from Cell 4
# ============================================================

# Total responses per donor (0-20)
# Most donors have few responses — right-skewed distribution
total_responses = np.random.choice(
    [0, 1, 2, 3, 4, 5, 10, 15, 20],
    n,
    p=[0.15, 0.25, 0.20, 0.15, 0.10, 0.07, 0.04, 0.02, 0.02]
)

# Age — based on real data (mean=32.9, std=4.5)
age = np.random.normal(loc=32.9, scale=4.5, size=n).clip(18, 65).astype(int)

# Gender — real data shows mostly male
gender = np.random.choice([0, 1], n, p=[0.75, 0.25])

# Blood type (1-8)
blood_type = np.random.choice(range(1, 9), n)

# Total lifetime donations (based on real: mean=3.08, max=10)
total_donations = np.random.choice(
    [0, 1, 2, 3, 4, 5, 7, 10],
    n,
    p=[0.10, 0.20, 0.20, 0.20, 0.15, 0.08, 0.04, 0.03]
)

# Days since last response
days_since_last = np.where(
    total_responses == 0,
    999,  # Never responded
    np.random.choice(
        [0, 5, 15, 30, 60, 90, 180, 365, 999],
        n,
        p=[0.15, 0.15, 0.15, 0.15, 0.12, 0.10, 0.08, 0.05, 0.05]
    )
)

# Hour of notification (0-23)
hour_of_day = np.random.choice(range(24), n)

# Day of week (0=Sunday, 6=Saturday)
day_of_week = np.random.choice(range(7), n)

# Urgency level (1=normal, 2=critical)
urgency_level = np.random.choice([1, 2], n, p=[0.70, 0.30])

# Distance from hospital (km)
distance_km = np.random.exponential(scale=8, size=n).clip(0.5, 50)

print("✅ Features generated")
print(f"   Total responses range: {total_responses.min()} - {total_responses.max()}")
print(f"   Age range:             {age.min()} - {age.max()}")
print(f"   Distance range:        {distance_km.min():.1f} - {distance_km.max():.1f} km")

✅ Features generated
   Total responses range: 0 - 20
   Age range:             20 - 46
   Distance range:        0.5 - 46.2 km


In [7]:
# ============================================================
# Step 2: Generate acceptance label based on real patterns
# Each factor affects the probability of accepting
# ============================================================

def calculate_acceptance_probability(i):
    """
    Calculate probability of accepting a blood request.
    Based on WHO blood donation research and real patterns.
    """

    # Start with real baseline from our data (47%)
    prob = 0.47

    # --- Factor 1: Past acceptance rate (most important) ---
    if total_responses[i] > 0:
        # Simulate past acceptance rate for this donor
        past_rate = np.random.beta(2, 2)  # Between 0 and 1
        prob += (past_rate - 0.5) * 0.40  # Weight: 40%

    # --- Factor 2: Recency (how recently they responded) ---
    days = days_since_last[i]
    if days == 999:
        prob -= 0.15   # Never responded → lower probability
    elif days <= 7:
        prob += 0.20   # Responded this week → very active
    elif days <= 30:
        prob += 0.10   # Responded this month → active
    elif days <= 90:
        prob += 0.00   # Neutral
    elif days <= 180:
        prob -= 0.05   # Getting inactive
    else:
        prob -= 0.15   # Very inactive

    # --- Factor 3: Urgency (critical requests get more response) ---
    if urgency_level[i] == 2:   # CRITICAL
        prob += 0.15
    else:                        # NORMAL
        prob -= 0.05

    # --- Factor 4: Distance (closer = more likely to come) ---
    dist = distance_km[i]
    if dist <= 3:
        prob += 0.15   # Very close
    elif dist <= 10:
        prob += 0.05   # Close
    elif dist <= 20:
        prob += 0.00   # Neutral
    elif dist <= 35:
        prob -= 0.10   # Far
    else:
        prob -= 0.20   # Very far

    # --- Factor 5: Time of day ---
    hour = hour_of_day[i]
    if 8 <= hour <= 20:
        prob += 0.05   # Daytime — more likely to see notification
    else:
        prob -= 0.10   # Night — less likely

    # --- Factor 6: Loyalty (more donations = more committed) ---
    donations = total_donations[i]
    if donations >= 5:
        prob += 0.10
    elif donations >= 2:
        prob += 0.05

    # --- Factor 7: Age ---
    a = age[i]
    if 25 <= a <= 45:
        prob += 0.05   # Prime donation age
    elif a > 55:
        prob -= 0.05   # Older → slightly less active

    # Clamp between 5% and 95% — nobody is 0% or 100%
    return np.clip(prob, 0.05, 0.95)

# Calculate probability for each row
probabilities = np.array([calculate_acceptance_probability(i) for i in range(n)])

# Convert probability to binary label (0 or 1)
accepted = (np.random.random(n) < probabilities).astype(int)

print("✅ Labels generated")
print(f"   Acceptance rate: {accepted.mean():.2%}  (real data: 47.06%)")
print(f"   Accepted:        {accepted.sum()}")
print(f"   Not accepted:    {(accepted == 0).sum()}")

✅ Labels generated
   Acceptance rate: 64.20%  (real data: 47.06%)
   Accepted:        321
   Not accepted:    179


In [8]:
# ============================================================
# Step 3: Build the final DataFrame
# ============================================================

df = pd.DataFrame({
    # Donor behavior features
    'total_responses':  total_responses,
    'days_since_last':  days_since_last,
    'total_donations':  total_donations,

    # Donor profile features
    'age':              age,
    'gender':           gender,
    'blood_type':       blood_type,

    # Request context features
    'urgency_level':    urgency_level,
    'distance_km':      distance_km.round(2),
    'hour_of_day':      hour_of_day,
    'day_of_week':      day_of_week,

    # Target label
    'accepted':         accepted,
})

# Derived features — same as DonorScoringService Rule-Based formula
df['acceptance_rate'] = np.where(
    df['total_responses'] > 0,
    # Simulate acceptance rate based on label distribution
    np.random.beta(
        df['accepted'] * 3 + 1,
        (1 - df['accepted']) * 3 + 1
    ),
    0.5  # Cold start default
)

df['recency_score'] = np.where(
    df['days_since_last'] == 999,
    0.0,
    np.exp(-df['days_since_last'] / 60)
)

df['loyalty_score'] = (df['total_donations'] / 10).clip(0, 1)

print("✅ DataFrame built")
print(f"   Shape: {df.shape}")
print(f"   Columns: {list(df.columns)}")
print()
print(df.head(10))

✅ DataFrame built
   Shape: (500, 14)
   Columns: ['total_responses', 'days_since_last', 'total_donations', 'age', 'gender', 'blood_type', 'urgency_level', 'distance_km', 'hour_of_day', 'day_of_week', 'accepted', 'acceptance_rate', 'recency_score', 'loyalty_score']

   total_responses  days_since_last  total_donations  age  gender  blood_type  \
0                1              180                3   34       0           2   
1               10               15                4   41       1           8   
2                3                5                3   37       1           7   
3                2                0                1   30       0           8   
4                1               60                2   28       1           1   
5                1               60                4   35       1           5   
6                0              999                3   26       0           8   
7                5                0                0   41       1           4   
8   

In [9]:
print("=== Dataset Quality Check ===")
print()

# 1. هل في قيم ناقصة؟
print("Missing values:")
print(df.isnull().sum())
print()

# 2. توزيع الـ Label
print("Label distribution:")
print(f"  Accepted (1):     {(df['accepted'] == 1).sum()} ({(df['accepted'] == 1).mean():.1%})")
print(f"  Not accepted (0): {(df['accepted'] == 0).sum()} ({(df['accepted'] == 0).mean():.1%})")
print()

# 3. هل المنطق صح؟ (acceptance_rate عالية = أكثر قبول؟)
high_rate = df[df['acceptance_rate'] > 0.7]['accepted'].mean()
low_rate  = df[df['acceptance_rate'] < 0.3]['accepted'].mean()
print(f"Acceptance rate > 0.7 → actual acceptance: {high_rate:.1%}  (يجب يكون عالي)")
print(f"Acceptance rate < 0.3 → actual acceptance: {low_rate:.1%}   (يجب يكون منخفض)")
print()

# 4. هل المسافة تؤثر صح؟
close   = df[df['distance_km'] < 5]['accepted'].mean()
far     = df[df['distance_km'] > 30]['accepted'].mean()
print(f"Distance < 5km  → acceptance: {close:.1%}  (يجب يكون أعلى)")
print(f"Distance > 30km → acceptance: {far:.1%}   (يجب يكون أدنى)")
print()

# 5. هل الاستعجال يؤثر صح؟
critical = df[df['urgency_level'] == 2]['accepted'].mean()
normal   = df[df['urgency_level'] == 1]['accepted'].mean()
print(f"Critical urgency → acceptance: {critical:.1%}  (يجب يكون أعلى)")
print(f"Normal urgency   → acceptance: {normal:.1%}   (يجب يكون أدنى)")

=== Dataset Quality Check ===

Missing values:
total_responses    0
days_since_last    0
total_donations    0
age                0
gender             0
blood_type         0
urgency_level      0
distance_km        0
hour_of_day        0
day_of_week        0
accepted           0
acceptance_rate    0
recency_score      0
loyalty_score      0
dtype: int64

Label distribution:
  Accepted (1):     321 (64.2%)
  Not accepted (0): 179 (35.8%)

Acceptance rate > 0.7 → actual acceptance: 100.0%  (يجب يكون عالي)
Acceptance rate < 0.3 → actual acceptance: 1.1%   (يجب يكون منخفض)

Distance < 5km  → acceptance: 69.3%  (يجب يكون أعلى)
Distance > 30km → acceptance: 52.9%   (يجب يكون أدنى)

Critical urgency → acceptance: 81.4%  (يجب يكون أعلى)
Normal urgency   → acceptance: 57.2%   (يجب يكون أدنى)


In [10]:
import os

# Create directory if not exists
os.makedirs('../data', exist_ok=True)

# Save full dataset
df.to_csv('../data/bloodbridge_dataset.csv', index=False)

# Save train/test split
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['accepted']  # Keep same ratio in both splits
)

train_df.to_csv('../data/train.csv', index=False)
test_df.to_csv('../data/test.csv',  index=False)

print("✅ Dataset saved:")
print(f"   Full dataset:  data/bloodbridge_dataset.csv  ({len(df)} rows)")
print(f"   Training set:  data/train.csv                ({len(train_df)} rows)")
print(f"   Test set:      data/test.csv                 ({len(test_df)} rows)")
print()
print(f"Train acceptance rate: {train_df['accepted'].mean():.1%}")
print(f"Test  acceptance rate: {test_df['accepted'].mean():.1%}")
print("(Should be similar — stratify=True ensures this)")

✅ Dataset saved:
   Full dataset:  data/bloodbridge_dataset.csv  (500 rows)
   Training set:  data/train.csv                (400 rows)
   Test set:      data/test.csv                 (100 rows)

Train acceptance rate: 64.2%
Test  acceptance rate: 64.0%
(Should be similar — stratify=True ensures this)


In [11]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# ============================================================
# Step 1: Define features and label
# ============================================================

# Features the model will learn from
feature_cols = [
    'total_responses',
    'days_since_last',
    'total_donations',
    'age',
    'gender',
    'blood_type',
    'urgency_level',
    'distance_km',
    'hour_of_day',
    'day_of_week',
    'acceptance_rate',
    'recency_score',
    'loyalty_score',
]

X = df[feature_cols]  # Features (input)
y = df['accepted']    # Label (output: 1 or 0)

# ============================================================
# Step 2: Split into training and test sets
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 80% training, 20% testing
    random_state=42,     # Same split every run
    stratify=y           # Keep same ratio in both splits
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples:     {len(X_test)}")
print(f"Features:         {len(feature_cols)}")

# ============================================================
# Step 3: Train XGBoost
# ============================================================

model = xgb.XGBClassifier(
    max_depth=4,              # Shallow trees — prevent overfitting
    learning_rate=0.1,        # Conservative learning rate
    n_estimators=100,         # 100 trees
    subsample=0.8,            # Use 80% of samples per tree
    colsample_bytree=0.8,     # Use 80% of features per tree
    scale_pos_weight=192/308, # Handle class imbalance
    eval_metric='logloss',
    random_state=42,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

print("✅ Model trained successfully")

Training samples: 400
Test samples:     100
Features:         13
✅ Model trained successfully


In [12]:
# ============================================================
# Step 4: Evaluate the model
# ============================================================

y_pred       = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

auc       = roc_auc_score(y_test, y_pred_proba)
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)

print("=== Model Performance ===")
print(f"AUC-ROC:   {auc:.3f}  (target: > 0.72)")
print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1 Score:  {f1:.3f}")
print()

# Verdict
if auc >= 0.72:
    print(f"✅ Model PASSED — AUC-ROC {auc:.3f} > 0.72")
else:
    print(f"⚠️  Model needs improvement — AUC-ROC {auc:.3f} < 0.72")

print()
print("=== Detailed Report ===")
print(classification_report(y_test, y_pred, target_names=['Not Accepted', 'Accepted']))

=== Model Performance ===
AUC-ROC:   0.926  (target: > 0.72)
Accuracy:  0.810
Precision: 0.857
Recall:    0.844
F1 Score:  0.850

✅ Model PASSED — AUC-ROC 0.926 > 0.72

=== Detailed Report ===
              precision    recall  f1-score   support

Not Accepted       0.73      0.75      0.74        36
    Accepted       0.86      0.84      0.85        64

    accuracy                           0.81       100
   macro avg       0.79      0.80      0.80       100
weighted avg       0.81      0.81      0.81       100



In [13]:
import matplotlib.pyplot as plt

# ============================================================
# Step 5: Feature importance — what does the model rely on?
# ============================================================

importance = pd.DataFrame({
    'feature':   feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("=== Feature Importance ===")
print(importance.to_string(index=False))
print()
print("Top 3 most important features:")
for _, row in importance.head(3).iterrows():
    print(f"  {row['feature']}: {row['importance']:.3f}")

=== Feature Importance ===
        feature  importance
acceptance_rate    0.441138
  urgency_level    0.087633
  recency_score    0.063764
days_since_last    0.059352
total_responses    0.046515
     blood_type    0.045777
    distance_km    0.042751
         gender    0.042192
    hour_of_day    0.038938
            age    0.038329
    day_of_week    0.034952
total_donations    0.034294
  loyalty_score    0.024365

Top 3 most important features:
  acceptance_rate: 0.441
  urgency_level: 0.088
  recency_score: 0.064


In [14]:
import pickle
import os

# Create models directory
os.makedirs('../models', exist_ok=True)

# Save model
with open('../models/donor_scorer.pkl', 'wb') as f:
    pickle.dump(model, f)

# Save feature names (important for FastAPI)
with open('../models/feature_names.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

# Save model metrics for documentation
import json
metrics = {
    'auc_roc':   round(auc, 4),
    'accuracy':  round(accuracy, 4),
    'precision': round(precision, 4),
    'recall':    round(recall, 4),
    'f1_score':  round(f1, 4),
    'feature_importance': importance.set_index('feature')['importance'].round(4).to_dict(),
    'training_samples': len(X_train),
    'test_samples':     len(X_test),
    'dataset_size':     len(df),
}

with open('../models/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("✅ Model saved:")
print(f"   models/donor_scorer.pkl    — XGBoost model")
print(f"   models/feature_names.pkl  — Feature names list")
print(f"   models/metrics.json       — Performance metrics")
print()
print("Model is ready for FastAPI integration")

✅ Model saved:
   models/donor_scorer.pkl    — XGBoost model
   models/feature_names.pkl  — Feature names list
   models/metrics.json       — Performance metrics

Model is ready for FastAPI integration


In [15]:
# Test on a donor similar to donor_id=5 from our real database
# donor 5: accepted=1, days_since_last=0, total_donations=7

test_donor = pd.DataFrame({
    'total_responses':  [1],
    'days_since_last':  [0],    # استجاب اليوم
    'total_donations':  [7],    # تبرع 7 مرات
    'age':              [34],
    'gender':           [0],    # ذكر
    'blood_type':       [2],
    'urgency_level':    [2],    # critical
    'distance_km':      [3.5],  # قريب
    'hour_of_day':      [10],   # الساعة 10 صباحاً
    'day_of_week':      [1],    # الاثنين
    'acceptance_rate':  [1.0],  # قبل كل مرة
    'recency_score':    [1.0],  # نشيط اليوم
    'loyalty_score':    [0.7],  # 7 تبرعات
})

probability = model.predict_proba(test_donor)[0][1]
decision    = "✅ يُرسل إشعار" if probability >= 0.5 else "❌ لا يُرسل إشعار"

print(f"Donor profile: نشيط + قريب + قبل دايماً + critical request")
print(f"Acceptance Probability: {probability:.2%}")
print(f"Decision: {decision}")
print()

# Test on inactive donor
inactive_donor = pd.DataFrame({
    'total_responses':  [1],
    'days_since_last':  [180],   # ما استجاب من 6 أشهر
    'total_donations':  [0],     # ما تبرع أبداً
    'age':              [25],
    'gender':           [0],
    'blood_type':       [5],
    'urgency_level':    [1],     # normal
    'distance_km':      [35.0],  # بعيد
    'hour_of_day':      [2],     # الساعة 2 فجراً
    'day_of_week':      [5],
    'acceptance_rate':  [0.0],   # ما قبل أبداً
    'recency_score':    [0.05],  # خامل جداً
    'loyalty_score':    [0.0],
})

prob2    = model.predict_proba(inactive_donor)[0][1]
decision2 = "✅ يُرسل إشعار" if prob2 >= 0.5 else "❌ لا يُرسل إشعار"

print(f"Donor profile: خامل + بعيد + ما قبل أبداً + normal request")
print(f"Acceptance Probability: {prob2:.2%}")
print(f"Decision: {decision2}")

Donor profile: نشيط + قريب + قبل دايماً + critical request
Acceptance Probability: 99.66%
Decision: ✅ يُرسل إشعار

Donor profile: خامل + بعيد + ما قبل أبداً + normal request
Acceptance Probability: 0.48%
Decision: ❌ لا يُرسل إشعار
